In [15]:
from pathlib import Path
from itertools import product
import pandas as pd

from overlapsim.data import SimulationData
from overlapsim.simulation import run_simulation
from overlapsim.metrics import summarise_results


# Project root and results directory
project_root = Path.cwd()
results_dir = project_root / "results"

# Running a single simulation

In [ ]:
# Initialise the simulation data class
gen = SimulationData(n_samples=500)

In [ ]:
# Run the simulation 10000 times. Will take about 5 mins on laptop.
results = run_simulation(gen, n_replications=1000)

In [10]:
summary = summarise_results(results)
print(summary)

mean_ate_bias          0.003866
rmse_ate               0.112330
mean_cate_rmse         0.300732
sd_ate_hat             0.112320
mean_treatment_rate    0.399960
dtype: float64


In [12]:
results

,ate_true,ate_hat,ate_bias,cate_rmse,treatment_rate,replication,n_samples,overlap_coef,treatment_prevalence,heterogeneity_coef
0,1.0,1.042185,0.042185,0.325097,0.388,0,500,1.0,0.4,0
1,1.0,1.049902,0.049902,0.210508,0.400,1,500,1.0,0.4,0
2,1.0,1.080360,0.080360,0.489741,0.370,2,500,1.0,0.4,0
3,1.0,0.786588,-0.213412,0.334160,0.410,3,500,1.0,0.4,0
4,1.0,1.015622,0.015622,0.126543,0.382,4,500,1.0,0.4,0
...,...,...,...,...,...,...,...,...,...,...
995,1.0,0.954356,-0.045644,0.189498,0.352,995,500,1.0,0.4,0
996,1.0,0.842762,-0.157238,0.293299,0.416,996,500,1.0,0.4,0
997,1.0,0.867720,-0.132280,0.359624,0.414,997,500,1.0,0.4,0
998,1.0,1.029319,0.029319,0.216097,0.384,998,500,1.0,0.4,0


# Running all simulations

In [17]:
heterogeneity_vals = [0, 0.5, 1, 2] # degree of heterogeneity
overlap_vals = [1, 3]   # degree of overlap
treatment_prev_vals = [0.1, 0.4]    # treatment prevalence
n_sample_vals = [300, 500, 1000, 2000]  # number of samples

all_results = []

# Run a simulation once for every combination in grid
for i, (heterogeneity_coef, overlap_coef, treatment_prevalence, n_samples) in enumerate(
    product(heterogeneity_vals, overlap_vals, treatment_prev_vals, n_sample_vals)
):
    # Display the simulation being run to keep track of where the loop is at
    print(
        f"[{i+1}/64] "
        f"heterogeneity={heterogeneity_coef}, "
        f"overlap={overlap_coef}, "
        f"treatment_prevalence={treatment_prevalence}, "
        f"n_samples={n_samples}"
    )

    # Initialise the data generating process with selected parameters
    gen = SimulationData(
        n_samples=n_samples,
        overlap_coef=overlap_coef,
        treatment_prevalence=treatment_prevalence,
        heterogeneity_coef=heterogeneity_coef,
    )

    # Run the simulation 1000 times per simulation
    results = run_simulation(
        generator=gen,
        n_replications=1000,
        base_seed=123 + 10000 * i,  # vary the base seed over simulations
    )

    # Append results over each loop to a single list
    all_results.append(results)

# Concatenate the results to a master dataset
full_results = pd.concat(all_results, ignore_index=True)

# Save the master dataset
full_results.to_csv(results_dir / "simulation_replication_results.csv", index=False)

full_results

[1/64] heterogeneity=0, overlap=1, treatment_prevalence=0.1, n_samples=300
[2/64] heterogeneity=0, overlap=1, treatment_prevalence=0.1, n_samples=500
[3/64] heterogeneity=0, overlap=1, treatment_prevalence=0.1, n_samples=1000
[4/64] heterogeneity=0, overlap=1, treatment_prevalence=0.1, n_samples=2000
[5/64] heterogeneity=0, overlap=1, treatment_prevalence=0.4, n_samples=300
[6/64] heterogeneity=0, overlap=1, treatment_prevalence=0.4, n_samples=500
[7/64] heterogeneity=0, overlap=1, treatment_prevalence=0.4, n_samples=1000
[8/64] heterogeneity=0, overlap=1, treatment_prevalence=0.4, n_samples=2000
[9/64] heterogeneity=0, overlap=3, treatment_prevalence=0.1, n_samples=300
[10/64] heterogeneity=0, overlap=3, treatment_prevalence=0.1, n_samples=500
[11/64] heterogeneity=0, overlap=3, treatment_prevalence=0.1, n_samples=1000
[12/64] heterogeneity=0, overlap=3, treatment_prevalence=0.1, n_samples=2000
[13/64] heterogeneity=0, overlap=3, treatment_prevalence=0.4, n_samples=300
[14/64] heterog

,ate_true,ate_hat,ate_bias,cate_rmse,treatment_rate,replication,n_samples,overlap_coef,treatment_prevalence,heterogeneity_coef
0,1.000000,0.323812,-0.676188,1.554708,0.090000,0,300,1,0.1,0.0
1,1.000000,0.681087,-0.318913,1.019739,0.126667,1,300,1,0.1,0.0
2,1.000000,1.116470,0.116470,0.990681,0.116667,2,300,1,0.1,0.0
3,1.000000,1.525379,0.525379,0.930370,0.093333,3,300,1,0.1,0.0
4,1.000000,1.858907,0.858907,1.928639,0.100000,4,300,1,0.1,0.0
...,...,...,...,...,...,...,...,...,...,...
63995,1.026843,1.669411,0.642568,0.798865,0.384500,995,2000,3,0.4,2.0
63996,0.989498,1.086480,0.096982,0.409114,0.384000,996,2000,3,0.4,2.0
63997,0.991734,0.780258,-0.211477,0.663217,0.400500,997,2000,3,0.4,2.0
63998,1.003455,0.670265,-0.333189,1.034117,0.402000,998,2000,3,0.4,2.0


In [18]:
# Apply the summary function to the dataset for each parameter combination
# Will create one row per parameter combination as a DataFrame
summary_table = (
    full_results
    .groupby(
        ["heterogeneity_coef", "overlap_coef", "treatment_prevalence", "n_samples"]
    )
    .apply(summarise_results)
    .reset_index()
)

# Save the summary table dataset
summary_table.to_csv(results_dir / "simulation_summary_results.csv", index=False)

summary_table

,heterogeneity_coef,overlap_coef,treatment_prevalence,n_samples,mean_ate_bias,rmse_ate,mean_cate_rmse,sd_ate_hat,mean_treatment_rate
0,0.0,1,0.1,300,2.617083,333.582477,120.519493,333.739122,0.100327
1,0.0,1,0.1,500,-0.014856,0.489899,0.923320,0.489919,0.099866
2,0.0,1,0.1,1000,0.000965,0.192684,0.477177,0.192778,0.099363
3,0.0,1,0.1,2000,0.004052,0.115333,0.289987,0.115320,0.099737
4,0.0,1,0.4,300,-0.003584,0.158375,0.427322,0.158414,0.400893
...,...,...,...,...,...,...,...,...,...
59,2.0,3,0.1,2000,-1.353728,18.839656,7.195589,18.801119,0.099911
60,2.0,3,0.4,300,-0.456699,4.491836,3.795894,4.469754,0.399863
61,2.0,3,0.4,500,-0.350776,3.943728,2.550314,3.929267,0.399788
62,2.0,3,0.4,1000,-0.111568,1.157280,1.422709,1.151715,0.400013
